In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper ,ArxivAPIWrapper
from langgraph.graph import StateGraph, START, END

In [3]:
load_dotenv(override=True)

llm = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.7
)

response = llm.invoke("Explain what LangSmith does in simple terms.")
print(response.content)

LangSmith is a tool that helps developers build and improve applications that use language models (like AI that understands and generates text). It makes it easier to track how these AI models perform, find mistakes, and fix them — so the apps work better and smarter. Think of it as a helpful assistant for creating and refining AI-powered language tools.


In [4]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)

def arxiv_tool_function(query:str)->str:
    print("Using Tool Arxiv")
    arxiv_runner = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
    return arxiv_runner.invoke(query)
    
arxiv = Tool(
    name="arxiv",
    description = "Searches Academic paper on Arxiv related to a company or topic.",
    func= arxiv_tool_function
)

In [5]:
def tavily_tool_func(query:str)->str:
    print("Using Tool Tavily")
    tavily_runner = TavilySearchResults(api_key = os.getenv("TAVILY_API_KEY"))
    return tavily_runner.invoke(query)

tavily = Tool(
    name ="tavily_search_tool",
    description="Perform a live web search for the latest financial and market information.",
    func=tavily_tool_func
)

In [6]:
api_wrapper_wikipedia = WikipediaAPIWrapper(top_k_results=2,doc_content_chars_max=500)
def wikipedia_tool_func(query:str)->str:
    print("using Wikipedia Tool")
    wikipedia_runner = WikipediaQueryRun(api_wrapper=api_wrapper_wikipedia)
    return wikipedia_runner.run(query)

wikipedia = Tool(
    name="wikipedia",
    description="Fetches background information about a company or topic from wikipedia",
    func=wikipedia_tool_func
)

In [7]:
def retriver_node(state):
    company = state["company"]
    
    print(f"\nGathering information for: {company}.\n")

    wikipedia_data = wikipedia.run(company)
    web_data = tavily.run(f"{company} financial performance 2025")
    research_data = arxiv.run(f"{company} machine learning finance application.")

    combined_data = f"""
    Wikipedia: {wikipedia_data}
    Web_search: {web_data}
    arxiv_research: {research_data}
    """

    return {"retrieved_data": combined_data}

In [8]:
def analyzer_node(state):
    prompt = f"""
    You are a financial research analyst.
    Analyze the following data and extract insights about financial performance, market trends, and risks.

    Data:
    {state['retrieved_data']}

    Provide a concise summary with key financial highlights, growth drivers, and risks.
    """

    analysis = llm.invoke(prompt)
    return {"analysis": analysis}

In [9]:
def writer_node(state):
    prompt = f"""
    Based on this analysis, generate a professional Investment Memo for {state['company']}.

    Include sections:
    - Company Overview
    - Key Financial Highlights
    - Market Trends
    - Risks & Challenges
    - Recommendation

    Use a confident and analytical tone.

    Analysis:
    {state['analysis']}
    """

    memo = llm.invoke(prompt)
    return {"memo": memo}

In [10]:
from typing import TypedDict

class Investment_Memo_State(TypedDict):
    company: str
    retrieved_data: str
    analysis: str
    memo: str

In [11]:
graph = StateGraph(Investment_Memo_State)
graph.add_node("Retriever",retriver_node)
graph.add_node("Analyser",analyzer_node)
graph.add_node("writer",writer_node)
graph.add_edge(START,"Retriever")
graph.add_edge("Retriever","Analyser")
graph.add_edge("Analyser","writer")
graph.add_edge("writer",END)

Investment_Memo_graph = graph.compile()

In [12]:
results = Investment_Memo_graph.invoke({"company":"Apple"})
print(results["memo"])


Gathering information for: Apple.

using Wikipedia Tool
Using Tool Tavily


/var/folders/jc/b5mkvz716lg36m165mxmg28m0000gn/T/ipykernel_27640/3945215928.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_runner = TavilySearchResults(api_key = os.getenv("TAVILY_API_KEY"))


Using Tool Arxiv
content='**Investment Memo: Apple Inc. (AAPL)**  \n*Date: Fiscal Year 2025 Review*\n\n---\n\n### Company Overview\n\nApple Inc. continues to solidify its position as a global technology leader, innovating across hardware, software, and services. Renowned for its flagship iPhone, Apple has successfully diversified its revenue streams through expanding services and complementary product lines including Mac, iPad, and wearables. This strategic breadth, coupled with a strong brand and ecosystem, supports sustained growth and market resilience. Apple’s focus on innovation is underscored by increasing R&D investments, positioning the company to capture new opportunities in technology and consumer engagement.\n\n---\n\n### Key Financial Highlights\n\n- **Revenue Growth:**  \n  Apple posted robust revenue gains, with Q3 2025 revenue at $94.0 billion (+10% YoY) and Q4 2025 revenue reaching $102.47 billion (+7.9% YoY). The full fiscal year 2025 revenue totaled $416.16 billion, m